In [ ]:
import pymc as pm; import arviz as az; import sys; import numpy as np; import pickle; import pytensor
import nutpie; import pytensor.tensor as pt; import matplotlib.pyplot as plt
sys.path.append(r'C:\Users\awast\OneDrive\Desktop\MKM')
from _CO_Oxidation.config import kb_eV, T, kb_J, h, F, N_A
from _CO_Oxidation import wrapper_base as wp
from _CO_Oxidation.wrapper_base import plot_posteriors, plot_model_fits, plot_coverages, plot_drc, simulate_fake_data

C_KOH_list = np.array([0.25, 0.5, 1]) # in M 
P_CO_list = 0.01*np.array([0.1, 1, 10, 100]) # in atm 
experiments_data = pickle.load(open('import_Ag90Pd10_base_replicates.pkl', 'rb'))
wp.process_experimental_data(experiments_data, C_KOH_list, P_CO_list)
E_in = wp.E_in; P_CO_in = wp.P_CO_in; C_KOH_in = wp.C_KOH_in

def fit_and_evaluate(model, draws=1000, tune=2000, chains=4, cores=4, init_mean=None, target_accept=0.9):
    compiled_model = nutpie.compile_pymc_model(model)
    trace = nutpie.sample(compiled_model, draws=draws, tune=tune, chains=chains, cores=cores, init_mean=init_mean, target_accept=target_accept)
    pm.compute_log_likelihood(trace, progressbar=False, model=model)
    loo = az.loo(trace, pointwise=True)
    print(loo); az.plot_khat(loo); plt.title("Pareto k diagnostic"); plt.show()
    trace = wp.add_post_sampling_observables(trace)
    return trace, loo

def observables(log_rate):
    pm.Deterministic('log_rate', log_rate)
    sigma_rel = pm.HalfNormal('sigma_rel', sigma=0.1) # 0.1
    rate = pm.LogNormal('rate', mu=log_rate, sigma=sigma_rel, observed=wp.rate_obs_matrix)
    return rate

Ag_comp = 0.9

# LH 
Errors too large. Bad fits. 

In [ ]:
with pm.Model() as LH:
    '''
    1. CO + * <-> CO* (QEA)
    2. CO* + OH* -> COOH* + * (RDS)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.Normal('deltaG4_0', mu=0.0, sigma=0.2)
    Gact2_0 = pm.TruncatedNormal('Gact2_0', mu=0.7, sigma=0.2, lower=0.0)

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    Gact2 = Gact2_0
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_k2 = np.log(kb_J*T/h) - Gact2/(kb_eV*T)

    term_CO = log_K1 + np.log(P_CO_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in)
    zeros = pt.zeros_like(term_CO)
    log_theta_Pd = - pt.logsumexp(pt.stack([zeros, term_CO, term_OH_Pd]), axis=0)
    log_theta_CO = term_CO + log_theta_Pd
    log_theta_OH_Pd = term_OH_Pd + log_theta_Pd

    theta_CO = pm.Deterministic('theta_CO', pt.exp(log_theta_CO))
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(log_theta_OH_Pd))
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', pt.exp(log_theta_Pd))

    # Rate expression
    log_rate = log_k2 + log_theta_CO + log_theta_OH_Pd + np.log(1-Ag_comp)
    rate = observables(log_rate) 

trace_LH, loo_LH = fit_and_evaluate(LH)

ppc_LH = plot_posteriors(trace_LH, LH)
plot_model_fits(trace_LH, ppc_LH, loo_LH); plot_coverages(trace_LH)

# BF w/o PCT
Errors too large. Bad fits.

In [ ]:
with pm.Model() as BF_wo_PCT:
    '''
    1. CO + * <-> CO* (QEA)
    2. CO* + OH# -> COOH* + # (RDS)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    5. OH- + # <-> OH# + (e-) (QEA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.Normal('deltaG4_0', mu=0.0, sigma=0.2) 
    deltaG5_0 = pm.Normal('deltaG5_0', mu=0.0, sigma=0.2) 
    Gact2_0 = pm.TruncatedNormal('Gact2_0', mu=0.7, sigma=0.2, lower=0.5) # Wrangling multimodality 

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    deltaG5 = deltaG5_0 - E_in
    Gact2 = Gact2_0
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_K5 = -deltaG5/(kb_eV*T)
    log_k2 = np.log(kb_J*T/h) - Gact2/(kb_eV*T)

    term_CO = log_K1 + np.log(P_CO_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in)
    term_OH_Ag = log_K5 + np.log(C_KOH_in)
    zeros = pt.zeros_like(term_CO)
    log_theta_Pd = - pt.logsumexp(pt.stack([zeros, term_CO, term_OH_Pd]), axis=0)
    log_theta_Ag = - pt.logsumexp(pt.stack([zeros, term_OH_Ag]), axis=0)
    log_theta_CO = term_CO + log_theta_Pd
    log_theta_OH_Pd = term_OH_Pd + log_theta_Pd
    log_theta_OH_Ag = term_OH_Ag + log_theta_Ag

    theta_CO = pm.Deterministic('theta_CO', pt.exp(log_theta_CO))
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(log_theta_OH_Pd))
    theta_OH_pound = pm.Deterministic('theta_OH_pound', pt.exp(log_theta_OH_Ag))
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', pt.exp(log_theta_Pd))
    theta_empty_Ag = pm.Deterministic('theta_empty_Ag', pt.exp(log_theta_Ag))

    # Rate expression
    log_rate = log_k2 + log_theta_CO + log_theta_OH_Ag + np.log(Ag_comp)
    rate = observables(log_rate) 

trace_BF_wo_PCT, loo_BF_wo_PCT = fit_and_evaluate(BF_wo_PCT, target_accept=0.95)

ppc_BF_wo_PCT = plot_posteriors(trace_BF_wo_PCT, BF_wo_PCT)
plot_model_fits(trace_BF_wo_PCT, ppc_BF_wo_PCT, loo_BF_wo_PCT); plot_coverages(trace_BF_wo_PCT)

# ER
Good simple model. 

In [ ]:
with pm.Model() as ER:
    '''
    1. CO + * <-> CO* (QEA)
    2. CO* + OH- -> COOH* + (e-) (RDS)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.Normal('deltaG4_0', mu=0.0, sigma=0.2)
    beta_2 = pm.Uniform('beta_2', lower=0.0, upper=1.0)   
    Gact2_0 = pm.TruncatedNormal('Gact2_0', mu=0.7, sigma=0.2, lower=0.0)

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    Gact2 = Gact2_0 - beta_2*E_in
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_k2 = np.log(kb_J*T/h) - Gact2/(kb_eV*T)

    term_CO = log_K1 + np.log(P_CO_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in)
    zeros = pt.zeros_like(term_CO)
    log_theta_Pd = - pt.logsumexp(pt.stack([zeros, term_CO, term_OH_Pd]), axis=0)
    log_theta_CO = term_CO + log_theta_Pd
    log_theta_OH_Pd = term_OH_Pd + log_theta_Pd

    theta_CO = pm.Deterministic('theta_CO', pt.exp(log_theta_CO))
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(log_theta_OH_Pd))
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', pt.exp(log_theta_Pd))

    # Rate expression
    log_rate = log_k2 + log_theta_CO + np.log(C_KOH_in)
    rate = observables(log_rate) 

trace_ER, loo_ER = fit_and_evaluate(ER)

ppc_ER = plot_posteriors(trace_ER, ER)
plot_model_fits(trace_ER, ppc_ER, loo_ER); plot_coverages(trace_ER)

# BF (!)
Good model. 

In [ ]:
with pm.Model() as BF:
    '''
    1. CO + * <-> CO* (QEA)
    2. CO* + OH(q-1)# -> COOH* + # + (1-q)(e-) (RDS) (BF)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    5. OH- + # <-> OH(q-1)# + (q)(e-) (QEA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.Normal('deltaG4_0', mu=0.0, sigma=0.2)
    deltaG5_0 = pm.Normal('deltaG5_0', mu=0.0, sigma=0.2) 
    beta_2 = pm.Uniform('beta_2', lower=0.0, upper=1.0)
    q = pm.Uniform('q', lower=0.0, upper=1.0)   
    Gact2_0 = pm.TruncatedNormal('Gact2_0', mu=0.7, sigma=0.2, lower=0.0)

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    deltaG5 = deltaG5_0 - q * E_in
    Gact2 = Gact2_0 - beta_2 * (1.0 - q) * E_in
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_K5 = -deltaG5/(kb_eV*T)
    log_k2 = np.log(kb_J*T/h) - Gact2/(kb_eV*T)

    term_CO = log_K1 + np.log(P_CO_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in)
    term_OH_Ag = log_K5 + np.log(C_KOH_in)
    zeros = pt.zeros_like(term_CO)
    log_theta_Pd = - pt.logsumexp(pt.stack([zeros, term_CO, term_OH_Pd]), axis=0)
    log_theta_Ag = - pt.logsumexp(pt.stack([zeros, term_OH_Ag]), axis=0)
    log_theta_CO = term_CO + log_theta_Pd
    log_theta_OH_Pd = term_OH_Pd + log_theta_Pd
    log_theta_OH_Ag = term_OH_Ag + log_theta_Ag

    theta_CO = pm.Deterministic('theta_CO', pt.exp(log_theta_CO))
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(log_theta_OH_Pd))
    theta_OH_pound = pm.Deterministic('theta_OH_pound', pt.exp(log_theta_OH_Ag))
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', pt.exp(log_theta_Pd))
    theta_empty_Ag = pm.Deterministic('theta_empty_Ag', pt.exp(log_theta_Ag))

    # Rate expression
    log_rate_BF = log_k2 + log_theta_CO + log_theta_OH_Ag + np.log(Ag_comp)
    log_rate = log_rate_BF
    rate = observables(log_rate) 

trace_BF, loo_BF = fit_and_evaluate(BF, target_accept=0.95)

ppc_BF = plot_posteriors(trace_BF, BF)
plot_model_fits(trace_BF, ppc_BF, loo_BF); plot_coverages(trace_BF)

# BF LH
LH has no DRC. Discard. 

In [ ]:
with pm.Model() as BF_LH:
    '''
    1. CO + * <-> CO* (QEA)
    2a. CO* + OH(q-1)# -> COOH* + # + (1-q)(e-) (RDS) (BF)
    2b. CO* + OH* -> COOH* + * (RDS)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    5. OH- + # <-> OH(q-1)# + (q)(e-) (QEA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.Normal('deltaG4_0', mu=0.0, sigma=0.2) 
    deltaG5_0 = pm.Normal('deltaG5_0', mu=0.0, sigma=0.2)
    beta_2 = pm.Uniform('beta_2', lower=0.0, upper=1.0) 
    q = pm.Uniform('q', lower=0.0, upper=1.0)
    Gact2_BF_0 = pm.TruncatedNormal('Gact2_BF_0', mu=0.7, sigma=0.2, lower=0.0)
    Gact2_LH_0 = pm.TruncatedNormal('Gact2_LH_0', mu=0.7, sigma=0.2, lower=0.0)

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    deltaG5 = deltaG5_0 - q * E_in
    Gact2_BF = Gact2_BF_0 - beta_2 * (1.0 - q) * E_in
    Gact2_LH = Gact2_LH_0
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_K5 = -deltaG5/(kb_eV*T)
    log_k2_BF = np.log(kb_J*T/h) - Gact2_BF/(kb_eV*T)
    log_k2_LH = np.log(kb_J*T/h) - Gact2_LH/(kb_eV*T)

    term_CO = log_K1 + np.log(P_CO_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in)
    term_OH_Ag = log_K5 + np.log(C_KOH_in)
    zeros = pt.zeros_like(term_CO)
    log_theta_Pd = - pt.logsumexp(pt.stack([zeros, term_CO, term_OH_Pd]), axis=0)
    log_theta_Ag = - pt.logsumexp(pt.stack([zeros, term_OH_Ag]), axis=0)
    log_theta_CO = term_CO + log_theta_Pd
    log_theta_OH_Pd = term_OH_Pd + log_theta_Pd
    log_theta_OH_Ag = term_OH_Ag + log_theta_Ag

    theta_CO = pm.Deterministic('theta_CO', pt.exp(log_theta_CO))
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(log_theta_OH_Pd))
    theta_OH_pound = pm.Deterministic('theta_OH_pound', pt.exp(log_theta_OH_Ag))
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', pt.exp(log_theta_Pd))
    theta_empty_Ag = pm.Deterministic('theta_empty_Ag', pt.exp(log_theta_Ag))

    # Rate expression
    log_rate_BF = log_k2_BF + log_theta_CO + log_theta_OH_Ag + np.log(Ag_comp)
    log_rate_LH = log_k2_LH + log_theta_CO + log_theta_OH_Pd + np.log(1-Ag_comp)
    log_rate = pt.logaddexp(log_rate_BF, log_rate_LH)
    rate = observables(log_rate) 

trace_BF_LH, loo_BF_LH = fit_and_evaluate(BF_LH, target_accept=0.99)

ppc_BF_LH  = plot_posteriors(trace_BF_LH , BF_LH)
plot_model_fits(trace_BF_LH, ppc_BF_LH, loo_BF_LH); plot_coverages(trace_BF_LH)
plot_drc(BF_LH, trace_BF_LH, perturb_vars=['Gact2_BF_0', 'Gact2_LH_0'], perturb_labels=['BF', 'LH'])

# BF ER (!!)
Higher ELPD. ER has significant DRC. 

In [ ]:
with pm.Model() as BF_ER:
    '''
    1. CO + * <-> CO* (QEA)
    2a. CO* + OH(q-1)# -> COOH* + # + (1-q)(e-) (RDS) (BF)
    2b. CO* + OH- -> COOH* + (e-) (RDS) (ER)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    5. OH- + # <-> OH(q-1)# + (q)(e-) (QEA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.Normal('deltaG4_0', mu=0.0, sigma=0.2)
    deltaG5_0 = pm.Normal('deltaG5_0', mu=0.0, sigma=0.2)
    beta_2_BF = pm.Uniform('beta_2_BF', lower=0.0, upper=1.0)
    beta_2_ER = pm.Uniform('beta_2_ER', lower=0.0, upper=1.0) # upper=0.4 
    q = pm.Uniform('q', lower=0.0, upper=1.0)
    Gact2_BF_0 = pm.TruncatedNormal('Gact2_BF_0', mu=0.7, sigma=0.2, lower=0.0)
    Gact2_ER_0 = pm.TruncatedNormal('Gact2_ER_0', mu=0.7, sigma=0.2, lower=0.0)

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    deltaG5 = deltaG5_0 - q * E_in
    Gact2_BF = Gact2_BF_0 - beta_2_BF * (1.0 - q) * E_in
    Gact2_ER = Gact2_ER_0 - beta_2_ER * E_in
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_K5 = -deltaG5/(kb_eV*T)
    log_k2_BF = np.log(kb_J*T/h) - Gact2_BF/(kb_eV*T)
    log_k2_ER = np.log(kb_J*T/h) - Gact2_ER/(kb_eV*T)

    term_CO = log_K1 + np.log(P_CO_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in)
    term_OH_Ag = log_K5 + np.log(C_KOH_in)
    zeros = pt.zeros_like(term_CO)
    log_theta_Pd = - pt.logsumexp(pt.stack([zeros, term_CO, term_OH_Pd]), axis=0)
    log_theta_Ag = - pt.logsumexp(pt.stack([zeros, term_OH_Ag]), axis=0)
    log_theta_CO = term_CO + log_theta_Pd
    log_theta_OH_Pd = term_OH_Pd + log_theta_Pd
    log_theta_OH_Ag = term_OH_Ag + log_theta_Ag

    theta_CO = pm.Deterministic('theta_CO', pt.exp(log_theta_CO))
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(log_theta_OH_Pd))
    theta_OH_pound = pm.Deterministic('theta_OH_pound', pt.exp(log_theta_OH_Ag))
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', pt.exp(log_theta_Pd))
    theta_empty_Ag = pm.Deterministic('theta_empty_Ag', pt.exp(log_theta_Ag))

    # Rate expression
    log_rate_BF = log_k2_BF + log_theta_CO + log_theta_OH_Ag + np.log(Ag_comp)
    log_rate_ER = log_k2_ER + log_theta_CO + np.log(C_KOH_in)
    log_rate = pt.logaddexp(log_rate_BF, log_rate_ER)
    rate = observables(log_rate) 

trace_BF_ER, loo_BF_ER = fit_and_evaluate(BF_ER, target_accept=0.95)

ppc_BF_ER = plot_posteriors(trace_BF_ER, BF_ER)
plot_model_fits(trace_BF_ER, ppc_BF_ER, loo_BF_ER); plot_coverages(trace_BF_ER)
plot_drc(BF_ER, trace_BF_ER, perturb_vars=['Gact2_BF_0', 'Gact2_ER_0'], perturb_labels=['BF', 'ER'])

# BF ER LH
ELPD increase not statistically significant. 

In [ ]:
with pm.Model() as BF_ER_LH:
    '''
    1. CO + * <-> CO* (QEA)
    2a. CO* + OH(q-1)# -> COOH* + # + (1-q)(e-) (RDS) (BF)
    2b. CO* + OH- -> COOH* + (e-) (RDS) (ER)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    5. OH- + # <-> OH(q-1)# + (q)(e-) (QEA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.Normal('deltaG4_0', mu=0.0, sigma=0.2)
    deltaG5_0 = pm.Normal('deltaG5_0', mu=0.0, sigma=0.2)
    beta_2_BF = pm.Uniform('beta_2_BF', lower=0.0, upper=1.0)
    beta_2_ER = pm.Uniform('beta_2_ER', lower=0.0, upper=1.0)
    q = pm.Uniform('q', lower=0.0, upper=1.0)
    Gact2_BF_0 = pm.TruncatedNormal('Gact2_BF_0', mu=0.7, sigma=0.2, lower=0.0)
    Gact2_ER_0 = pm.TruncatedNormal('Gact2_ER_0', mu=0.7, sigma=0.2, lower=0.0)
    Gact2_LH_0 = pm.TruncatedNormal('Gact2_LH_0', mu=0.7, sigma=0.2, lower=0.0)

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    deltaG5 = deltaG5_0 - q * E_in
    Gact2_BF = Gact2_BF_0 - beta_2_BF * (1.0 - q) * E_in
    Gact2_ER = Gact2_ER_0 - beta_2_ER * E_in
    Gact2_LH = Gact2_LH_0
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_K5 = -deltaG5/(kb_eV*T)
    log_k2_BF = np.log(kb_J*T/h) - Gact2_BF/(kb_eV*T)
    log_k2_ER = np.log(kb_J*T/h) - Gact2_ER/(kb_eV*T)
    log_k2_LH = np.log(kb_J*T/h) - Gact2_LH/(kb_eV*T)

    term_CO = log_K1 + np.log(P_CO_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in)
    term_OH_Ag = log_K5 + np.log(C_KOH_in)
    zeros = pt.zeros_like(term_CO)
    log_theta_Pd = - pt.logsumexp(pt.stack([zeros, term_CO, term_OH_Pd]), axis=0)
    log_theta_Ag = - pt.logsumexp(pt.stack([zeros, term_OH_Ag]), axis=0)
    log_theta_CO = term_CO + log_theta_Pd
    log_theta_OH_Pd = term_OH_Pd + log_theta_Pd
    log_theta_OH_Ag = term_OH_Ag + log_theta_Ag

    theta_CO = pm.Deterministic('theta_CO', pt.exp(log_theta_CO))
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(log_theta_OH_Pd))
    theta_OH_pound = pm.Deterministic('theta_OH_pound', pt.exp(log_theta_OH_Ag))
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', pt.exp(log_theta_Pd))
    theta_empty_Ag = pm.Deterministic('theta_empty_Ag', pt.exp(log_theta_Ag))

    # Rate expression
    log_rate_BF = log_k2_BF + log_theta_CO + log_theta_OH_Ag + np.log(Ag_comp)
    log_rate_ER = log_k2_ER + log_theta_CO + np.log(C_KOH_in)
    log_rate_LH = log_k2_LH + log_theta_CO + log_theta_OH_Pd + np.log(1-Ag_comp)
    log_rate = pt.logsumexp(pt.stack([log_rate_BF, log_rate_ER, log_rate_LH]), axis=0)
    rate = observables(log_rate) 

trace_BF_ER_LH, loo_BF_ER_LH = fit_and_evaluate(BF_ER_LH, target_accept=0.99)

ppc_BF_ER_LH = plot_posteriors(trace_BF_ER_LH, BF_ER_LH)
plot_model_fits(trace_BF_ER_LH, ppc_BF_ER_LH, loo_BF_ER_LH); plot_coverages(trace_BF_ER_LH)
plot_drc(BF_ER_LH, trace_BF_ER_LH, perturb_vars=['Gact2_BF_0', 'Gact2_ER_0', 'Gact2_LH_0'], perturb_labels=['BF', 'ER', 'LH'])

# CO BF ER
ELPD increase not statistically significant. 

In [ ]:
with pm.Model() as CO_BF_ER:
    '''
    1. CO + * <-> CO* (SSA)
    2a. CO* + OH(q-1)# -> COOH* + # + (1-q)(e-) (SSA)
    2b. CO* + OH- -> COOH* + * + (e-) (SSA)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    5. OH- + # <-> OH(q-1)# + (q)(e-) (QEA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.Normal('deltaG4_0', mu=0.0, sigma=0.2) 
    deltaG5_0 = pm.Normal('deltaG5_0', mu=0.0, sigma=0.2) 
    beta_2_BF = pm.Uniform('beta_2_BF', lower=0.0, upper=1.0)
    beta_2_ER = pm.Uniform('beta_2_ER', lower=0.0, upper=1.0)
    q = pm.Uniform('q', lower=0.0, upper=1.0)
    Gact1_0 = pm.TruncatedNormal('Gact1_0', mu=0.7, sigma=0.2, lower=0.0)
    Gact2_BF_0 = pm.TruncatedNormal('Gact2_BF_0', mu=0.7, sigma=0.2, lower=0.0)
    Gact2_ER_0 = pm.TruncatedNormal('Gact2_ER_0', mu=0.7, sigma=0.2, lower=0.0)

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    deltaG5 = deltaG5_0 - q * E_in
    Gact1 = Gact1_0
    Gact2_BF = Gact2_BF_0 - beta_2_BF * (1.0 - q) * E_in
    Gact2_ER = Gact2_ER_0 - beta_2_ER * E_in
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_K5 = -deltaG5/(kb_eV*T)
    log_k1 = np.log(kb_J*T/h) - Gact1/(kb_eV*T)
    log_k2_BF = np.log(kb_J*T/h) - Gact2_BF/(kb_eV*T)
    log_k2_ER = np.log(kb_J*T/h) - Gact2_ER/(kb_eV*T)

    term_OH_Ag = log_K5 + np.log(C_KOH_in)
    zeros = pt.zeros_like(term_OH_Ag)
    log_theta_Ag = -pt.logsumexp(pt.stack([zeros, term_OH_Ag]), axis=0)
    log_theta_OH_Ag = term_OH_Ag + log_theta_Ag
    term_OH_Pd = log_K4 + np.log(C_KOH_in)
    log_A_Pd = pt.logsumexp(pt.stack([zeros, term_OH_Pd]), axis=0) 
    log_k1_PCO = log_k1 + np.log(P_CO_in)                    
    log_k_minus_1 = log_k1 - log_K1                                
    log_k_BF_app = log_k2_BF + log_theta_OH_Ag + np.log(Ag_comp)     
    log_k_ER_app = log_k2_ER + np.log(C_KOH_in)
    log_denom_Pd = pt.logsumexp(pt.stack([log_k1_PCO, log_A_Pd + log_k_minus_1, log_A_Pd + log_k_BF_app, log_A_Pd + log_k_ER_app]), axis=0)

    log_theta_CO = log_k1_PCO - log_denom_Pd
    log_theta_Pd = pt.logsumexp(pt.stack([log_k_minus_1 + zeros, log_k_BF_app, log_k_ER_app]), axis=0) - log_denom_Pd 
    log_theta_OH_Pd = term_OH_Pd + log_theta_Pd

    theta_CO = pm.Deterministic('theta_CO', pt.exp(log_theta_CO))
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(log_theta_OH_Pd))
    theta_OH_pound = pm.Deterministic('theta_OH_pound', pt.exp(log_theta_OH_Ag))
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', pt.exp(log_theta_Pd))
    theta_empty_Ag = pm.Deterministic('theta_empty_Ag', pt.exp(log_theta_Ag))

    # Rate expression
    log_k_total_app = pt.logaddexp(log_k_BF_app, log_k_ER_app)
    log_rate = log_k_total_app + log_theta_CO
    rate = observables(log_rate) 

trace_CO_BF_ER, loo_CO_BF_ER = fit_and_evaluate(CO_BF_ER)

ppc_CO_BF_ER = plot_posteriors(trace_CO_BF_ER, CO_BF_ER)
plot_model_fits(trace_CO_BF_ER, ppc_CO_BF_ER, loo_CO_BF_ER); plot_coverages(trace_CO_BF_ER)
plot_drc(CO_BF_ER, trace_CO_BF_ER, perturb_vars=['Gact2_BF_0', 'Gact2_ER_0', 'Gact1_0'], perturb_labels=['BF', 'ER', 'CO ads'])

# BF OH
Comparable ELPD with CO BF ER. CO ads energy is low. 

In [ ]:
with pm.Model() as BF_OH:
    '''
    1. CO + * <-> CO* (QEA)
    2. CO* + OH(q-1)# -> COOH* + # + (1-q)(e-) (SSA)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    5. OH- + # <-> OH(q-1)# + (q)(e-) (SSA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.TruncatedNormal('deltaG4_0', mu=0.0, sigma=0.2, lower=0.0) # Forced to be larger than 0 eV to wrangle multimodality 
    deltaG5_0 = pm.Normal('deltaG5_0', mu=0.0, sigma=0.2)
    beta_2 = pm.Uniform('beta_2', lower=0.0, upper=1.0)
    beta_5 = pm.Uniform('beta_5', lower=0.0, upper=1.0)
    q = pm.Uniform('q', lower=0.0, upper=1.0)
    Gact2_0 = pm.TruncatedNormal('Gact2_0', mu=0.7, sigma=0.2, lower=0.0)
    Gact5_0 = pm.TruncatedNormal('Gact5_0', mu=0.7, sigma=0.2, lower=0.0)

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    deltaG5 = deltaG5_0 - q * E_in
    Gact2 = Gact2_0 - beta_2 * (1.0 - q) * E_in
    Gact5 = Gact5_0 - beta_5 * q * E_in
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_K5 = -deltaG5/(kb_eV*T)
    log_k2 = np.log(kb_J*T/h) - Gact2/(kb_eV*T)
    log_k5 = np.log(kb_J*T/h) - Gact5/(kb_eV*T)

    log_k_minus_5 = log_k5 - log_K5 
    log_k5_OH = log_k5 + np.log(C_KOH_in) 
    term_CO = log_K1 + np.log(P_CO_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in) 
    zeros = pt.zeros_like(term_OH_Pd) 
    log_denom_Pd = pt.logsumexp(pt.stack([zeros, term_CO, term_OH_Pd]), axis=0) 
    log_theta_CO = term_CO - log_denom_Pd
    log_theta_OH_Pd = term_OH_Pd - log_denom_Pd
    log_theta_empty_Pd = zeros - log_denom_Pd
    log_drain_Ag = log_k2 + log_theta_CO + np.log(1.0 - Ag_comp)
    log_denom_Ag = pt.logsumexp(pt.stack([log_k5_OH, log_k_minus_5, log_drain_Ag]), axis=0)
    log_theta_OH_Ag = log_k5_OH - log_denom_Ag
    log_theta_empty_Ag = pt.logsumexp(pt.stack([log_k_minus_5, log_drain_Ag]), axis=0) - log_denom_Ag
    log_k_app = log_k2 + log_theta_OH_Ag + np.log(Ag_comp) 
    
    theta_CO = pm.Deterministic('theta_CO', pt.exp(log_theta_CO)) 
    theta_OH_pound = pm.Deterministic('theta_OH_pound', pt.exp(log_theta_OH_Ag)) 
    theta_empty_Ag = pm.Deterministic('theta_empty_Ag', pt.exp(log_theta_empty_Ag)) 
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', pt.exp(log_theta_empty_Pd)) 
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(log_theta_OH_Pd)) 
    
    # Rate expression 
    log_rate = log_k_app + log_theta_CO 
    rate = observables(log_rate)

trace_BF_OH, loo_BF_OH = fit_and_evaluate(BF_OH)

ppc_BF_OH = plot_posteriors(trace_BF_OH, BF_OH)
plot_model_fits(trace_BF_OH, ppc_BF_OH, loo_BF_OH); plot_coverages(trace_BF_OH)
plot_drc(BF_OH, trace_BF_OH, perturb_vars=['Gact2_0', 'Gact5_0'], perturb_labels=['BF', 'OH ads'])

# BF ER OH (?)
Very high ELPD, but p_loo too high. CO ads energy too low, CO coverage too low at low pressures. 

In [ ]:
with pm.Model() as BF_ER_OH:
    '''
    1. CO + * <-> CO* (QEA)
    2a. CO* + OH(q-1)# -> COOH* + # + (1-q)(e-) (SSA)
    2b. CO* + OH- -> COOH* + * + (e-) (SSA)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    5. OH- + # <-> OH(q-1)# + (q)(e-) (SSA)
    '''
    # Priors
    deltaG1_0 = pm.Normal('deltaG1_0', mu=-0.4, sigma=0.2)
    deltaG4_0 = pm.TruncatedNormal('deltaG4_0', mu=0.0, sigma=0.2, lower=0.0) # Forced to be larger than 0 eV to wrangle multimodality 
    deltaG5_0 = pm.Normal('deltaG5_0', mu=0.0, sigma=0.2)
    beta_2_BF = pm.Uniform('beta_2_BF', lower=0.0, upper=1.0)
    beta_2_ER = pm.Uniform('beta_2_ER', lower=0.0, upper=1.0)
    beta_5 = pm.Uniform('beta_5', lower=0.0, upper=1.0)
    q = pm.Uniform('q', lower=0.0, upper=1.0)
    Gact2_BF_0 = pm.TruncatedNormal('Gact2_BF_0', mu=0.7, sigma=0.2, lower=0.0)
    Gact2_ER_0 = pm.TruncatedNormal('Gact2_ER_0', mu=0.7, sigma=0.2, lower=0.0)
    Gact5_0 = pm.TruncatedNormal('Gact5_0', mu=0.7, sigma=0.2, lower=0.0)

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    deltaG5 = deltaG5_0 - q * E_in
    Gact2_BF = Gact2_BF_0 - beta_2_BF * (1.0 - q) * E_in
    Gact2_ER = Gact2_ER_0 - beta_2_ER * E_in
    Gact5 = Gact5_0 - beta_5 * q * E_in
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_K5 = -deltaG5/(kb_eV*T)
    log_k2_BF = np.log(kb_J*T/h) - Gact2_BF/(kb_eV*T)
    log_k2_ER = np.log(kb_J*T/h) - Gact2_ER/(kb_eV*T)
    log_k5 = np.log(kb_J*T/h) - Gact5/(kb_eV*T)

    log_k_minus_5 = log_k5 - log_K5 
    log_k5_OH = log_k5 + np.log(C_KOH_in) 
    term_CO = log_K1 + np.log(P_CO_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in) 
    zeros = pt.zeros_like(term_OH_Pd) 
    log_denom_Pd = pt.logsumexp(pt.stack([zeros, term_CO, term_OH_Pd]), axis=0) 
    log_theta_CO = term_CO - log_denom_Pd
    log_theta_OH_Pd = term_OH_Pd - log_denom_Pd
    log_theta_empty_Pd = zeros - log_denom_Pd
    log_drain_Ag = log_k2_BF + log_theta_CO + np.log(1.0 - Ag_comp)
    log_denom_Ag = pt.logsumexp(pt.stack([log_k5_OH, log_k_minus_5, log_drain_Ag]), axis=0)
    log_theta_OH_Ag = log_k5_OH - log_denom_Ag
    log_theta_empty_Ag = pt.logsumexp(pt.stack([log_k_minus_5, log_drain_Ag]), axis=0) - log_denom_Ag
    log_k_BF_app = log_k2_BF + log_theta_OH_Ag + np.log(Ag_comp) 
    log_k_ER_app = log_k2_ER + np.log(C_KOH_in) 
    
    theta_CO = pm.Deterministic('theta_CO', pt.exp(log_theta_CO)) 
    theta_OH_pound = pm.Deterministic('theta_OH_pound', pt.exp(log_theta_OH_Ag)) 
    theta_empty_Ag = pm.Deterministic('theta_empty_Ag', pt.exp(log_theta_empty_Ag)) 
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', pt.exp(log_theta_empty_Pd)) 
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(log_theta_OH_Pd)) 
    
    # Rate expression 
    log_k_total_app = pt.logaddexp(log_k_BF_app, log_k_ER_app) 
    log_rate = log_k_total_app + log_theta_CO 
    rate = observables(log_rate)

trace_BF_ER_OH, loo_BF_ER_OH = fit_and_evaluate(BF_ER_OH)

ppc_CO_BF_ER_OH = plot_posteriors(trace_BF_ER_OH, BF_ER_OH)
plot_model_fits(trace_BF_ER_OH, ppc_CO_BF_ER_OH, loo_BF_ER_OH); plot_coverages(trace_BF_ER_OH)
plot_drc(BF_ER_OH, trace_BF_ER_OH, perturb_vars=['Gact2_BF_0', 'Gact2_ER_0', 'Gact5_0'], perturb_labels=['BF', 'ER', 'OH ads'])

# CO BF OH
Collapses to BF OH. CO ads has zero DRC. Discard. 

In [ ]:
with pm.Model() as CO_BF_OH:
    '''
    1. CO + * <-> CO* (SSA)
    2. CO* + OH(q-1)# -> COOH* + # + (1-q)(e-) (SSA)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    5. OH- + # <-> OH(q-1)# + (q)(e-) (SSA)
    '''
    # Priors
    deltaG1_0 = pm.TruncatedNormal('deltaG1_0', mu=-0.4, sigma=0.2, lower=-0.1, upper=0.0) # Wrangling multimodality to high ELPD regions 
    deltaG4_0 = pm.TruncatedNormal('deltaG4_0', mu=0.0, sigma=0.2, upper=0.3) # Wrangling multimodality
    deltaG5_0 = pm.TruncatedNormal('deltaG5_0', mu=0.0, sigma=0.2, upper=0.3) # Wrangling multimodality
    beta_2 = pm.Uniform('beta_2', lower=0.0, upper=0.1) # Wrangling multimodality 
    beta_5 = pm.Uniform('beta_5', lower=0.0, upper=1.0) 
    q = pm.Uniform('q', lower=0.0, upper=1.0) 
    Gact1_0 = pm.TruncatedNormal('Gact1_0', mu=0.7, sigma=0.2, lower=0.0) 
    Gact2_0 = pm.TruncatedNormal('Gact2_0', mu=0.7, sigma=0.2, lower=0.5, upper=0.8) # Wrangling multimodality by limiting upper bound to 0.8 eV
    Gact5_0 = pm.TruncatedNormal('Gact5_0', mu=0.7, sigma=0.2, lower=0.5, upper=0.8) # Wrangling multimodality by limiting upper bound to 0.8 eV

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    deltaG5 = deltaG5_0 - q * E_in
    Gact1 = Gact1_0
    Gact2 = Gact2_0 - beta_2 * (1.0 - q) * E_in
    Gact5 = Gact5_0 - beta_5 * q * E_in
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_K5 = -deltaG5/(kb_eV*T)
    log_k1 = np.log(kb_J*T/h) - Gact1/(kb_eV*T)
    log_k2 = np.log(kb_J*T/h) - Gact2/(kb_eV*T)
    log_k5 = np.log(kb_J*T/h) - Gact5/(kb_eV*T)

    log_k_minus_1 = log_k1 - log_K1 
    log_k_minus_5 = log_k5 - log_K5 
    log_k1_PCO = log_k1 + np.log(P_CO_in) 
    log_k5_OH = log_k5 + np.log(C_KOH_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in) 
    zeros = pt.zeros_like(term_OH_Pd)
    log_A_Pd = pt.logsumexp(pt.stack([zeros, term_OH_Pd]), axis=0) 

    k1_PCO = pt.exp(log_k1_PCO)
    k_minus_1 = pt.exp(log_k_minus_1)
    k_drain_Pd = pt.exp(log_k2 + np.log(Ag_comp))
    k_drain_Ag = pt.exp(log_k2 + np.log(1.0 - Ag_comp))
    A_Pd = pt.exp(log_A_Pd)
    k5_OH = pt.exp(log_k5_OH)
    k_minus_5 = pt.exp(log_k_minus_5)

    C1 = k1_PCO / A_Pd
    C2 = C1 + k_minus_1
    C3 = k_drain_Pd
    D1 = k5_OH
    D2 = k5_OH + k_minus_5
    D3 = k_drain_Ag
    a = C2 * D3
    b = C2 * D2 + C3 * D1 - C1 * D3
    c = -C1 * D2
    X = (-b + pt.sqrt(b**2 - 4 * a * c)) / (2 * a)
    Y = D1 / (D2 + D3 * X)

    log_theta_CO = pt.log(X)
    log_theta_OH_Ag = pt.log(Y)
    log_k_app = log_k2 + log_theta_OH_Ag + np.log(Ag_comp) 

    theta_CO = pm.Deterministic('theta_CO', X) 
    theta_OH_pound = pm.Deterministic('theta_OH_pound', Y) 
    theta_empty_Ag = pm.Deterministic('theta_empty_Ag', 1.0 - Y) 
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', (1.0 - X) / A_Pd) 
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(term_OH_Pd) * theta_empty_Pd) 
    
    # Rate expression 
    log_rate = log_k_app + log_theta_CO 
    rate = observables(log_rate)

trace_CO_BF_OH, loo_CO_BF_OH = fit_and_evaluate(CO_BF_OH, target_accept=0.95, tune=400, draws=600)

ppc_CO_BF_OH = plot_posteriors(trace_CO_BF_OH, CO_BF_OH)
plot_model_fits(trace_CO_BF_OH, ppc_CO_BF_OH, loo_CO_BF_OH); plot_coverages(trace_CO_BF_OH)
plot_drc(CO_BF_OH, trace_CO_BF_OH, perturb_vars=['Gact2_0', 'Gact1_0', 'Gact5_0'], perturb_labels=['BF', 'CO ads', 'OH ads'])

# CO BF ER OH
Collapses to BF_ER_OH. CO ads DRC has zero DRC. Discard.

In [ ]:
with pm.Model() as CO_BF_ER_OH:
    '''
    1. CO + * <-> CO* (SSA)
    2a. CO* + OH(q-1)# -> COOH* + # + (1-q)(e-) (SSA)
    2b. CO* + OH- -> COOH* + * + (e-) (SSA)
    3. COOH* + OH- -> CO2 + H2O + * + (e-) (Fast)
    4. OH- + * <-> OH* + (e-) (QEA)
    5. OH- + # <-> OH(q-1)# + (q)(e-) (SSA)
    '''
    # Priors
    deltaG1_0 = pm.TruncatedNormal('deltaG1_0', mu=-0.4, sigma=0.2, lower=-0.3, upper=0.0) # Wrangling multimodality
    deltaG4_0 = pm.TruncatedNormal('deltaG4_0', mu=0.0, sigma=0.2, lower=0.0, upper=0.3) # Wrangling multimodality, low sigma version has bounds (0.0, 0.3)
    deltaG5_0 = pm.TruncatedNormal('deltaG5_0', mu=0.0, sigma=0.2, lower=-0.3, upper=0.3) # Wrangling multimodality
    beta_2_BF = pm.Uniform('beta_2_BF', lower=0.0, upper=1.0)
    beta_2_ER = pm.Uniform('beta_2_ER', lower=0.0, upper=1.0)
    beta_5 = pm.Uniform('beta_5', lower=0.0, upper=1)
    q = pm.Uniform('q', lower=0.0, upper=1.0) 
    Gact1_0 = pm.TruncatedNormal('Gact1_0', mu=0.7, sigma=0.2, lower=0.0)
    Gact2_BF_0 = pm.TruncatedNormal('Gact2_BF_0', mu=0.7, sigma=0.2, lower=0.5, upper=0.8) # Wrangling multimodality by limiting upper bound to 0.8 eV
    Gact2_ER_0 = pm.TruncatedNormal('Gact2_ER_0', mu=0.7, sigma=0.2, lower=0.5, upper=0.8) # Wrangling multimodality by limiting upper bound to 0.8 eV
    Gact5_0 = pm.TruncatedNormal('Gact5_0', mu=0.7, sigma=0.2, lower=0.5, upper=0.8) # Wrangling multimodality by limiting upper bound to 0.8 eV

    # Thermodynamics 
    deltaG1 = deltaG1_0
    deltaG4 = deltaG4_0 - E_in
    deltaG5 = deltaG5_0 - q * E_in
    Gact1 = Gact1_0
    Gact2_BF = Gact2_BF_0 - beta_2_BF * (1.0 - q) * E_in
    Gact2_ER = Gact2_ER_0 - beta_2_ER * E_in
    Gact5 = Gact5_0 - beta_5 * q * E_in
    log_K1 = -deltaG1/(kb_eV*T)
    log_K4 = -deltaG4/(kb_eV*T)
    log_K5 = -deltaG5/(kb_eV*T)
    log_k1 = np.log(kb_J*T/h) - Gact1/(kb_eV*T)
    log_k2_BF = np.log(kb_J*T/h) - Gact2_BF/(kb_eV*T)
    log_k2_ER = np.log(kb_J*T/h) - Gact2_ER/(kb_eV*T)
    log_k5 = np.log(kb_J*T/h) - Gact5/(kb_eV*T)

    log_k_minus_1 = log_k1 - log_K1 
    log_k_minus_5 = log_k5 - log_K5 
    log_k1_PCO = log_k1 + np.log(P_CO_in) 
    log_k5_OH = log_k5 + np.log(C_KOH_in)
    term_OH_Pd = log_K4 + np.log(C_KOH_in) 
    zeros = pt.zeros_like(term_OH_Pd)
    log_A_Pd = pt.logsumexp(pt.stack([zeros, term_OH_Pd]), axis=0) 

    k1_PCO = pt.exp(log_k1_PCO)
    k_minus_1 = pt.exp(log_k_minus_1)
    k_ER_app = pt.exp(log_k2_ER + np.log(C_KOH_in))
    k_drain_Pd = pt.exp(log_k2_BF + np.log(Ag_comp))
    k_drain_Ag = pt.exp(log_k2_BF + np.log(1.0 - Ag_comp))
    A_Pd = pt.exp(log_A_Pd)
    k5_OH = pt.exp(log_k5_OH)
    k_minus_5 = pt.exp(log_k_minus_5)

    C1 = k1_PCO / A_Pd
    C2 = C1 + k_minus_1 + k_ER_app
    C3 = k_drain_Pd
    D1 = k5_OH
    D2 = k5_OH + k_minus_5
    D3 = k_drain_Ag
    a = C2 * D3
    b = C2 * D2 + C3 * D1 - C1 * D3
    c = -C1 * D2
    X = (-b + pt.sqrt(b**2 - 4 * a * c)) / (2 * a)
    Y = D1 / (D2 + D3 * X)

    log_theta_CO = pt.log(X)
    log_theta_OH_Ag = pt.log(Y)
    log_k_BF_app = log_k2_BF + log_theta_OH_Ag + np.log(Ag_comp) 
    log_k_ER_app = log_k2_ER + np.log(C_KOH_in) 

    theta_CO = pm.Deterministic('theta_CO', X) 
    theta_OH_pound = pm.Deterministic('theta_OH_pound', Y) 
    theta_empty_Ag = pm.Deterministic('theta_empty_Ag', 1.0 - Y) 
    theta_empty_Pd = pm.Deterministic('theta_empty_Pd', (1.0 - X) / A_Pd) 
    theta_OH_star = pm.Deterministic('theta_OH_star', pt.exp(term_OH_Pd) * theta_empty_Pd) 
    
    # Rate expression 
    log_k_total_app = pt.logaddexp(log_k_BF_app, log_k_ER_app) 
    log_rate = log_k_total_app + log_theta_CO 
    rate = observables(log_rate)

trace_CO_BF_ER_OH, loo_CO_BF_ER_OH = fit_and_evaluate(CO_BF_ER_OH, target_accept=0.95, tune=400, draws=600)

ppc_CO_BF_ER_OH = plot_posteriors(trace_CO_BF_ER_OH, CO_BF_ER_OH)
plot_model_fits(trace_CO_BF_ER_OH, ppc_CO_BF_ER_OH, loo_CO_BF_ER_OH); plot_coverages(trace_CO_BF_ER_OH)
plot_drc(CO_BF_ER_OH, trace_CO_BF_ER_OH, perturb_vars=['Gact2_BF_0', 'Gact2_ER_0', 'Gact1_0', 'Gact5_0'], perturb_labels=['BF', 'ER', 'CO ads', 'OH ads'])

# Comparison

In [ ]:
comparison_dict = {
    "BF": loo_BF,
    "BF_ER": loo_BF_ER,
    # "BF_ER_OH": loo_BF_ER_OH,
}

comp_df = az.compare(comparison_dict, ic="loo", method="stacking")
print(comp_df)
az.plot_compare(comp_df, insample_dev=False)
plt.show()

In [ ]:
!jupyter nbconvert --to html Ag90Pd10_base.ipynb